# MTA – Aufschreibung bereinigen

Batch-Version auf Basis von Skript 1 mit zusätzlichen Bereinigungsregeln.

In [1]:
# MTA – Aufschreibung bereinigen
#
# Ausgangsbasis: Skript 1
# Ergänzungen:
# - Verarbeitung einer frei pflegbaren Dateiliste
# - Dauer Arbeitszeit auf 0 Nachkommastellen runden
# - "Menge Gesamt (Stück)" in "MengeGesamtNIO" umbenennen
# - Zeilen löschen, wenn Bemerkung UND Station/OP leer sind
# - Mengen-Spalten N.i.O. / i.O. L4 / i.O. L5 von NULL auf 0 setzen
# - Dauer-Anlagen-Ausfall-Spalten von NULL auf 0 setzen
# - Station/OP-Splitting aus Skript 1 bleibt erhalten

import re
import unicodedata
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# =========================================================
# DATEIEN / PARAMETER


In [ ]:
# =========================================================
SHEET_NAME = "Aufschreibung"

# Hier die zu verarbeitenden Dateien eintragen.
# Beispiel: weitere Dateien einfach als neue Path(...)-Zeile ergänzen.
DATEIEN_LISTE = [
    Path(r"../data/raw/Störliste STW-Mittelteilanlage 2023_NEU.xlsx"),
    # Path(r"../data/raw/mta2024to2026/STW-Mittelteilanlage 2024.xlsx"),
    # Path(r"../data/raw/mta2024to2026/Störliste STW-Mittelteilanlage 2025.xlsx"),
    # Path(r"../data/raw/mta2024to2026/Störliste STW-Mittelteilanlage 2026.xlsx"),
]

OUT_DIR = Path(r"../data/processed")
OUT_BASENAME = "aufschreibung_mta_clean_gesamt"

# Falls du nachvollziehen willst, aus welcher Datei eine Zeile kommt: True setzen.
ADD_SOURCE_FILE_COLUMN = False


In [ ]:
# =========================================================
# 1) Spalten / Text Helpers


In [ ]:
# =========================================================
def _normalize_colname(c: object) -> str:
    c = "" if c is None else str(c)
    c = unicodedata.normalize("NFKC", c)
    c = c.replace("\n", " ")
    c = re.sub(r"\s+", " ", c).strip()
    c = re.sub(r"[‐-‒–—―]", "-", c)
    c = re.sub(r"\s*/\s*", "/ ", c)
    c = re.sub(r"\s+", " ", c).strip()
    return c


_CANON_PATTERNS = [
    (r"^station\s*/\s*op$", "Station/ OP"),
    (r"^station/op$", "Station/ OP"),
    (r"^datum\s*neu$", "DatumNEU"),
    (r"^zeit\s*von$", "Zeit von"),
    (r"^zeit\s*bis$", "Zeit bis"),
    (r"^unterbrechungsursache$", "Unterbrechungsursache"),
    (r"^bemerkung$", "Bemerkung"),
    (r"^dauer\s*org-?\s*mangel$", "Dauer Org-Mangel"),
]


def canonicalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = [_normalize_colname(c) for c in df.columns]
    canon = []

    for c in cols:
        c2 = c
        for pat, repl in _CANON_PATTERNS:
            if re.match(pat, c2, flags=re.IGNORECASE):
                c2 = repl
                break
        canon.append(c2)

    # Doppelte Spaltennamen abfangen
    seen = {}
    out = []
    for c in canon:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}_{seen[c]}")

    df = df.copy()
    df.columns = out
    return df


def find_col(cols, patterns) -> str | None:
    """Wie in Skript 1: robustes Finden der Station/OP-Spalte inkl. Fallback."""
    for pat in patterns:
        for c in cols:
            if re.match(pat, c, flags=re.IGNORECASE):
                return c

    # fallback nur für Station/OP-Suche
    for c in cols:
        lc = c.lower()
        if "station" in lc and "op" in lc:
            return c
    return None


def find_col_by_patterns(cols, patterns) -> str | None:
    """Allgemeine Spaltensuche ohne Station/OP-Fallback."""
    for pat in patterns:
        for c in cols:
            if re.match(pat, c, flags=re.IGNORECASE):
                return c
    return None


def normalize_free_text(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.map(lambda x: unicodedata.normalize("NFKC", x) if pd.notna(x) else x)
    s = s.str.lower()
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.str.replace(r"[‐-‒–—―]", "-", regex=True)
    s = s.str.replace(r"[•·●]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "k.a.": pd.NA, "k. a.": pd.NA})
    return s


def fuzzy_standardize(norm_s: pd.Series, threshold: int = 97, min_count: int = 2):
    # Vereinheitlicht NUR sehr ähnliche Schreibweisen (Typo/Spacing).
    counts = norm_s.dropna().value_counts()
    variants = counts.index.tolist()

    mapping = {}
    for v in variants:
        if v in mapping:
            continue
        mapping[v] = v
        matches = process.extract(
            v,
            variants,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=threshold,
            limit=None
        )
        for m, score, _ in matches:
            if m not in mapping:
                mapping[m] = v

    std = norm_s.map(mapping).astype("string")
    std = std.where(norm_s.notna(), pd.NA)

    map_df = pd.DataFrame({
        "original": list(mapping.keys()),
        "standard": list(mapping.values()),
        "count": [counts.get(k, 0) for k in mapping.keys()]
    }).sort_values(["standard", "count"], ascending=[True, False])

    if min_count > 1:
        map_df = map_df[map_df["count"] >= min_count].copy()

    return std, map_df


In [ ]:
# =========================================================
# 2) Datum / Zeit Helpers


In [ ]:
# =========================================================
def parse_excel_date_to_date(s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(s, errors="coerce")
    return dt.dt.normalize()


def parse_excel_time_to_str(s: pd.Series) -> pd.Series:
    def conv(x):
        if pd.isna(x):
            return pd.NA
        if isinstance(x, datetime.time):
            return x.strftime("%H:%M:%S")
        if isinstance(x, (pd.Timestamp, datetime.datetime)):
            return x.time().strftime("%H:%M:%S")
        if isinstance(x, (float, int, np.floating, np.integer)):
            seconds = int(round(float(x) * 24 * 3600)) % (24 * 3600)
            h = seconds // 3600
            m = (seconds % 3600) // 60
            sec = seconds % 60
            return f"{h:02d}:{m:02d}:{sec:02d}"
        txt = str(x).strip()
        if not txt:
            return pd.NA
        t = pd.to_datetime(txt, errors="coerce")
        if pd.isna(t):
            return pd.NA
        return t.time().strftime("%H:%M:%S")

    return s.map(conv).astype("string")


def time_str_to_minutes(s: pd.Series) -> pd.Series:
    def conv(x):
        if pd.isna(x):
            return np.nan
        parts = str(x).split(":")
        if len(parts) < 2:
            return np.nan
        h = int(parts[0])
        m = int(parts[1])
        sec = int(parts[2]) if len(parts) > 2 else 0
        return h * 60 + m + sec / 60

    return s.map(conv).astype(float)


In [ ]:
# =========================================================
# 3) Station/OP split Helpers aus Skript 1


In [ ]:
# =========================================================
def _clean_station_token(token: object) -> str:
    t = unicodedata.normalize("NFKC", str(token))
    t = t.strip()
    t = re.sub(r"(?i)\b(R|OP)\.\b", r"\1 ", t)
    t = re.sub(r"(?i)\b(R|OP)\.", r"\1 ", t)
    t = re.sub(r"(?i)\b(R|OP)\s*([0-9])", r"\1 \2", t)
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"(?i)^\s*op\b", "OP", t)
    t = re.sub(r"(?i)^\s*r\b", "R", t)
    return t


def split_station_op_simple(x: object) -> list[str]:
    if pd.isna(x):
        return []
    t = _clean_station_token(x)
    t = re.sub(r"[,/]", "|", t)
    parts = [_clean_station_token(p) for p in t.split("|")]
    return [p for p in parts if p and p.lower() not in ("nan", "none")]


def split_station_op_mta(x: object) -> list[str]:
    # MTA-Spezial:
    # - Trennung bei , oder /
    # - Zusätzlich: wenn in einem Chunk ein 2. 'R' oder 'OP' auftaucht, beginnt ein neuer Wert.
    if pd.isna(x):
        return []

    t = _clean_station_token(x)
    t = re.sub(r"[,/]", "|", t)
    chunks = [c.strip() for c in t.split("|") if c.strip()]

    out = []
    for ch in chunks:
        matches = list(re.finditer(r"(?i)\b(?:R|OP)\b", ch))
        if len(matches) <= 1:
            out.append(_clean_station_token(ch))
        else:
            pos = [m.start() for m in matches]
            for i, p in enumerate(pos):
                end = pos[i + 1] if i + 1 < len(pos) else len(ch)
                seg = ch[p:end].strip()
                if seg:
                    out.append(_clean_station_token(seg))

    seen = set()
    final = []
    for v in out:
        if v and v not in seen:
            seen.add(v)
            final.append(v)
    return final


def expand_split_columns(
    df: pd.DataFrame,
    source_col: str,
    splitter,
    prefix: str = "Station/ OP"
) -> pd.DataFrame:
    lists = df[source_col].map(splitter)
    max_len = int(lists.map(len).max()) if len(lists) else 0

    df2 = df.copy()
    df2[f"{prefix}_raw"] = df2[source_col].astype("string")

    for i in range(max_len):
        df2[f"{prefix}_{i + 1}"] = lists.map(
            lambda L: L[i] if len(L) > i else pd.NA
        ).astype("string")

    if max_len > 0:
        df2[source_col] = df2[f"{prefix}_1"]
    else:
        df2[source_col] = df2[source_col].astype("string")

    return df2


def drop_rows_empty_from(df: pd.DataFrame, start_col: str) -> tuple[pd.DataFrame, list[str]]:
    cols = list(df.columns)
    start_idx = cols.index(start_col)
    cols_from = cols[start_idx:]

    df2 = df.copy()
    for c in cols_from:
        if df2[c].dtype == object or str(df2[c].dtype).startswith("string"):
            df2[c] = df2[c].astype("string").str.strip()
            df2.loc[df2[c].isin(["", "nan", "NaN", "None"]), c] = pd.NA

    keep = df2[cols_from].notna().any(axis=1)
    return df2.loc[keep].copy(), cols_from


In [ ]:
# =========================================================
# 4) Neue fachliche Bereinigungsregeln


In [ ]:
# =========================================================
def _to_numeric_series(s: pd.Series) -> pd.Series:
    # Robust für Zahlen, die evtl. als Text mit deutschem Dezimalkomma vorliegen.
    if s.dtype == object or str(s.dtype).startswith("string"):
        s = s.astype("string").str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def _is_empty_value_series(s: pd.Series) -> pd.Series:
    txt = s.astype("string").str.strip()
    return s.isna() | txt.isna() | txt.isin(["", "nan", "NaN", "None", "<NA>"])


def fill_nulls_with_zero(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in columns:
        if c in df.columns:
            df[c] = _to_numeric_series(df[c]).fillna(0.0)
    return df


def apply_custom_rules(df: pd.DataFrame, station_col: str) -> pd.DataFrame:
    df = df.copy()

    # 1) Dauer Arbeitszeit runden: 59.9999 -> 60.0
    arbeitszeit_col = find_col_by_patterns(
        df.columns,
        [r"^Dauer\s+Arbeits-?\s*zeit$"]
    )
    if arbeitszeit_col:
        df[arbeitszeit_col] = _to_numeric_series(df[arbeitszeit_col]).round(0)

    # 2) Menge Gesamt (Stück) -> MengeGesamtNIO
    menge_gesamt_col = find_col_by_patterns(
        df.columns,
        [r"^Menge\s+Gesamt\s*\(\s*Stück\s*\)$", r"^Menge\s+Gesamt.*$"]
    )
    if menge_gesamt_col and menge_gesamt_col != "MengeGesamtNIO":
        df = df.rename(columns={menge_gesamt_col: "MengeGesamtNIO"})

    # 3) Mengen-Spalten von NULL auf 0 setzen
    mengen_cols = []
    for patterns in [
        [r"^Menge\s+N\.?\s*i\.?\s*O\.?$"],
        [r"^Menge\s+i\.?\s*O\.?\s*L4$"],
        [r"^Menge\s+i\.?\s*O\.?\s*L5$"],
    ]:
        c = find_col_by_patterns(df.columns, patterns)
        if c:
            mengen_cols.append(c)
    df = fill_nulls_with_zero(df, mengen_cols)

    # 4) Alle Dauer-Anlagen-Ausfall-Spalten von NULL auf 0 setzen
    # Enthält z. B. "Dauer Anlagen-Ausfall" und "Dauer Anlagen-Ausfall intern".
    dauer_anlagen_ausfall_cols = [
        c for c in df.columns
        if re.match(r"^Dauer\s+Anlagen[-\s]*Ausfall", c, flags=re.IGNORECASE)
    ]
    df = fill_nulls_with_zero(df, dauer_anlagen_ausfall_cols)

    # 5) Zeilen löschen, wenn Bemerkung UND Station/OP leer sind
    bemerkung_col = find_col_by_patterns(df.columns, [r"^Bemerkung$"])
    if bemerkung_col:
        drop_mask = _is_empty_value_series(df[bemerkung_col]) & _is_empty_value_series(df[station_col])
        df = df.loc[~drop_mask].copy()

    return df


In [ ]:
# =========================================================
# 5) Einzeldatei bereinigen


In [ ]:
# =========================================================
def bereinige_datensatz(df_raw: pd.DataFrame, source_file: Path | None = None) -> pd.DataFrame:
    df = canonicalize_columns(df_raw)

    # Spalten robust finden
    station_col = find_col(df.columns, [r"^Station/\s*OP$"])
    if station_col is None:
        raise ValueError("Spalte 'Station/ OP' nicht gefunden. Bitte Spaltennamen prüfen.")

    date_col = find_col_by_patterns(df.columns, [r"^DatumNEU$"])
    t_from_col = find_col_by_patterns(df.columns, [r"^Zeit von$"])
    t_to_col = find_col_by_patterns(df.columns, [r"^Zeit bis$"])

    # Original-Logik aus Skript 1: Zeilen entfernen, die ab Station/OP komplett leer sind
    df, _ = drop_rows_empty_from(df, station_col)

    # Datum / Zeit konvertieren
    if date_col:
        df[date_col] = parse_excel_date_to_date(df[date_col])

    if t_from_col:
        df[t_from_col] = parse_excel_time_to_str(df[t_from_col])
        df["Zeit_von_min"] = time_str_to_minutes(df[t_from_col])

    if t_to_col:
        df[t_to_col] = parse_excel_time_to_str(df[t_to_col])
        df["Zeit_bis_min"] = time_str_to_minutes(df[t_to_col])

    # Station/OP aufspalten (MTA-Regeln aus Skript 1)
    df = expand_split_columns(
        df,
        source_col=station_col,
        splitter=split_station_op_mta,
        prefix="Station/ OP"
    )

    # Neue Regeln anwenden
    df = apply_custom_rules(df, station_col=station_col)

    if ADD_SOURCE_FILE_COLUMN and source_file is not None:
        df["Quelle_Datei"] = source_file.name

    return df


In [ ]:
# =========================================================
# 6) Alle Dateien verarbeiten + zusammenführen


In [ ]:
# =========================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)

bereinigte_datenframes = []

for datei in DATEIEN_LISTE:
    print(f"Verarbeite Datei: {datei}")
    df_raw = pd.read_excel(datei, sheet_name=SHEET_NAME)
    df_bereinigt = bereinige_datensatz(df_raw, source_file=datei)
    bereinigte_datenframes.append(df_bereinigt)

if not bereinigte_datenframes:
    raise ValueError("DATEIEN_LISTE ist leer. Bitte mindestens eine Datei eintragen.")

df_gesamt = pd.concat(
    bereinigte_datenframes,
    axis=0,
    ignore_index=True
)

# Optional sortieren
if "DatumNEU" in df_gesamt.columns:
    df_gesamt = (
        df_gesamt
        .sort_values(by="DatumNEU")
        .reset_index(drop=True)
    )

# Freitext vereinheitlichen auf dem gesamten kombinierten Datensatz.
# Dadurch entstehen konsistente Standards über alle Dateien hinweg.
for free_col in ["Bemerkung", "Unterbrechungsursache"]:
    if free_col in df_gesamt.columns:
        df_gesamt[f"{free_col}_norm"] = normalize_free_text(df_gesamt[free_col])
        df_gesamt[f"{free_col}_std"], map_df = fuzzy_standardize(
            df_gesamt[f"{free_col}_norm"],
            threshold=97,
            min_count=2
        )

        safe = re.sub(r"[^a-z0-9]+", "_", free_col.lower())
        mapping_path = OUT_DIR / f"mta_mapping_{safe}.xlsx"
        map_df.to_excel(mapping_path, index=False)
        print(f"Mapping gespeichert: {mapping_path}")

print(df_gesamt.head(10))
print("Bereinigt gesamt:", df_gesamt.shape)


In [ ]:
# =========================================================
# 7) Export


In [ ]:
# =========================================================
OUT_CSV = OUT_DIR / f"{OUT_BASENAME}.csv"
OUT_XLSX = OUT_DIR / f"{OUT_BASENAME}.xlsx"

df_gesamt.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
df_gesamt.to_excel(OUT_XLSX, index=False)

print("Gespeichert:", OUT_CSV, "und", OUT_XLSX)
